In [ ]:
import requests
import pandas as pd
import os
from dotenv import load_dotenv
load_dotenv()
API_KEY = os.getenv("API_KEY")
URL = "https://api.gamebrain.co/v1/games?query=medieval+strategy+games"


def fetch_data():
    headers = {"x-api-key": API_KEY}
    response = requests.get(URL, headers=headers)
    response.raise_for_status()
    return response.json()


def transform(data):
    results = data["results"]

    rows = []
    for game in results:
        rows.append({
            "game_id": game.get("id"),
            "name": game.get("name"),
            "year": game.get("year"),
            "genre": game.get("genre"),
            "rating_mean": game.get("rating", {}).get("mean"),
            "rating_count": game.get("rating", {}).get("count"),
            "adult_only": game.get("adult_only"),
            "query": data.get("query"),
        })

    df = pd.DataFrame(rows)

    #  Correct ingestion timestamp
    df["ingestion_date"] = pd.Timestamp.utcnow()

    #  Dynamic partitions
    df["year_partition"] = df["ingestion_date"].dt.year
    df["month_partition"] = df["ingestion_date"].dt.month

    #  Arrow-safe types
    df = df.copy()
    df = df.convert_dtypes()
    df["ingestion_date"] = pd.to_datetime(df["ingestion_date"]).dt.tz_localize(None)

    return df


def save_parquet(df):
    year = df["year_partition"].iloc[0]
    month = df["month_partition"].iloc[0]

    base_path = f"data/games/year={year}/month={month:02d}"
    os.makedirs(base_path, exist_ok=True)

    file_path = f"{base_path}/games_{year}_{month:02d}.parquet"

    df.to_parquet(file_path, engine="pyarrow", index=False)

    print(f"Saved: {file_path}")


if __name__ == "__main__":
    data = fetch_data()
    df = transform(data)
    save_parquet(df)

Saved: data/games/year=2026/month=04/games_2026_04.parquet


C:\Users\felha\AppData\Local\Temp\ipykernel_17128\79310592.py:36: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  df["ingestion_date"] = pd.Timestamp.utcnow()


In [5]:
import os
print(os.getcwd())

c:\Users\felha\PYTHON\Gitdata\game-data-pipeline\game-data\scr


In [6]:
import requests
import pandas as pd
import os
from dotenv import load_dotenv
from datetime import datetime, timedelta
import numpy as np

load_dotenv()
API_KEY = os.getenv("API_KEY")
BASE_URL = "https://api.gamebrain.co/v1/games"

QUERY = "medieval strategy games"
PAGE_SIZE = 10   # API limit
MAX_PAGES = 7    # Number of fake months/backfill pages

# Starting fake month: 2026-04
START_YEAR = 2026
START_MONTH = 4

def fetch_page(query, offset, limit=PAGE_SIZE):
    headers = {"x-api-key": API_KEY}
    params = {
        "query": query,
        "offset": offset,
        "limit": limit,
        "sort": "computed_rating",
        "sort-order": "desc"
    }
    response = requests.get(BASE_URL, headers=headers, params=params)
    response.raise_for_status()
    return response.json().get("results", [])

def transform(results, fake_year, fake_month):
    rows = []
    for game in results:
        rows.append({
            "game_id": game.get("id"),
            "name": game.get("name"),
            "year": game.get("year"),
            "genre": game.get("genre"),
            "rating_mean": game.get("rating", {}).get("mean"),
            "rating_count": game.get("rating", {}).get("count"),
            "adult_only": game.get("adult_only"),
        })
    df = pd.DataFrame(rows)

    # Randomize ingestion_date per row within the month
    num_rows = len(df)
    start_date = datetime(fake_year, fake_month, 1)
    # End date is last day of month
    if fake_month == 12:
        end_date = datetime(fake_year+1, 1, 1) - timedelta(seconds=1)
    else:
        end_date = datetime(fake_year, fake_month+1, 1) - timedelta(seconds=1)

    random_dates = pd.to_datetime(
        np.random.randint(int(start_date.timestamp()), int(end_date.timestamp()), num_rows),
        unit='s'
    )
    df["ingestion_date"] = random_dates
    df["year_partition"] = fake_year
    df["month_partition"] = fake_month

    return df

def save_parquet(df):
    year = df["year_partition"].iloc[0]
    month = df["month_partition"].iloc[0]

    base_path = f"data/games/year={year}/month={month:02d}"
    os.makedirs(base_path, exist_ok=True)

    file_path = f"{base_path}/games_{year}_{month:02d}.parquet"
    df.to_parquet(file_path, engine="pyarrow", index=False)
    print(f"Saved: {file_path} ({len(df)} rows)")

# -------------------------------
# Simulated incremental backfill using pagination
# -------------------------------
if __name__ == "__main__":
    for page in range(MAX_PAGES):
        offset = page * PAGE_SIZE
        print(f"Fetching page {page+1} (offset={offset})")

        results = fetch_page(QUERY, offset)
        if not results:
            print("No more results from API")
            break

        # Calculate fake month backfill
        fake_year = START_YEAR if START_MONTH - page > 0 else START_YEAR - 1
        fake_month = START_MONTH - page if START_MONTH - page > 0 else 12 + (START_MONTH - page)

        df = transform(results, fake_year, fake_month)
        save_parquet(df)

    print("Backfill simulation complete!")

Fetching page 1 (offset=0)
Saved: data/games/year=2026/month=04/games_2026_04.parquet (10 rows)
Fetching page 2 (offset=10)
Saved: data/games/year=2026/month=03/games_2026_03.parquet (10 rows)
Fetching page 3 (offset=20)
Saved: data/games/year=2026/month=02/games_2026_02.parquet (10 rows)
Fetching page 4 (offset=30)
Saved: data/games/year=2026/month=01/games_2026_01.parquet (10 rows)
Fetching page 5 (offset=40)
Saved: data/games/year=2025/month=12/games_2025_12.parquet (10 rows)
Fetching page 6 (offset=50)
Saved: data/games/year=2025/month=11/games_2025_11.parquet (10 rows)
Fetching page 7 (offset=60)
Saved: data/games/year=2025/month=10/games_2025_10.parquet (10 rows)
Backfill simulation complete!
